# Project Janus — Compliance-RAG Explainer Model
## Fine-Tuning Qwen3-4B-Instruct via Unsloth (4-bit QLoRA)
**Team:** Cipher Ops | **SIH 2026:** SIH26160 (NTRO)

This turnkey notebook executes QLoRA fine-tuning on a free Google Colab T4 GPU (~45 minutes run time, <4GB VRAM usage).
Exports directly to **GGUF (`q4_k_m`)** for 100% offline local serving on the demo laptop.

In [ ]:
# 1. Install Unsloth & dependencies on Colab
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes

In [ ]:
# 2. Load 4-bit Qwen3-4B-Instruct base model
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen3-4B-unsloth-bnb-4bit",
    max_seq_length=max_seq_length,
    dtype=None,
    load_in_4bit=True,
)

# 3. Configure LoRA target modules
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=32,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

In [ ]:
# 4. Format Dataset with ChatML template
from unsloth.chat_templates import get_chat_template
from datasets import load_dataset

tokenizer = get_chat_template(tokenizer, chat_template="chatml")

def formatting_prompts_func(examples):
    convos = examples["messages"]
    texts = [tokenizer.apply_chat_template(c, tokenize=False, add_generation_prompt=False) for c in convos]
    return {"text": texts}

# Upload or clone instruct_dataset.jsonl from repository
dataset = load_dataset("json", data_files="instruct_dataset.jsonl", split="train")
dataset = dataset.map(formatting_prompts_func, batched=True)

In [ ]:
# 5. Execute Training
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=10,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=3407,
        output_dir="outputs",
    ),
)

trainer_stats = trainer.train()

In [ ]:
# 6. Export to GGUF (q4_k_m) for offline laptop serving
model.save_pretrained_gguf("janus-qwen3-4b", tokenizer, quantization_method="q4_k_m")
print("Exported: janus-qwen3-4b-q4_k_m.gguf ready for llama.cpp / Ollama!")